# D1.5 · Agent telemetry as a data source

**Function D — The Agentic SOC → The Agentic SOC — Detection**  ·  *Security of AI*

Builds on **[D1.4 · Detection engineering *for* agents](https://spbreed.github.io/cyber-commons/lessons/D1.4.html)**.

| | |
|---|---|
| Tools used | OpenTelemetry, OpenSearch |

## What this lesson is

**What it covers.** Ship OTEL agent traces into OpenSearch and query them.

**Why a security engineer needs it.** Prompts, traces, tool calls and approvals never reach the SIEM. The control it builds is: onboard agent telemetry deliberately; decide retention.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

You cannot detect on telemetry that was never emitted. Prompts, tool calls, decisions and identities are the four things an agent has to emit to be observable at all — and none of them appear in a standard application log.

> **At CyberTravels.** You cannot detect on what CyberTravels never emitted. Prompts, tool calls, decisions and identities are the four things missing from every application log CyberTravels has. R10, R11.

## 2 · The framework

```
   what an agent must emit to be observable at all

   +------------+  +-------------+  +-----------+  +------------+
   |  prompts   |  | tool calls  |  | decisions |  | identities |
   +------------+  +-------------+  +-----------+  +------------+
        |               |                |              |
        +---------------+----------------+--------------+
                                v
                   none of this is in an application log
                   retention is expensive and the cost is real
```

Agent telemetry has a property no other log source has: it contains the
**reasoning**, not just the action. The trace records what the agent was trying
to do, what it considered, and what the verifier said.

That is enormously useful for investigation and it is a retention and privacy
problem, because reasoning traces contain whatever was in the context window —
which routinely includes customer data, source code and secrets that were read
legitimately.

So retention has to be decided **per field**, not per record:

| Field | Forensic value | Sensitivity |
|---|---|---|
| timestamps, tool, target | high | low |
| verifier detail | high | low |
| acting identity + chain | high | low |
| model prompts | medium | **high** |
| tool results | high | **high** |

The first three are cheap and should be kept long. The last two are where the
retention conversation actually is.

## 3 · The procedure, as a skill

The run record contains a payment-card pattern, in a source file the agent read legitimately. The skill scans every field, then sets retention per field so timestamps and verdicts survive for 400 days and prompts do not survive 30.

### The skill — [`skills/detection/agent-telemetry-retention/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/detection/agent-telemetry-retention/SKILL.md)

```yaml
name: agent-telemetry-retention
description: >-
  Scan an agent's run record for sensitive content it read legitimately, then
  set retention per field rather than per record so the investigable parts
  survive and the prompts do not. Use when agent telemetry is being kept,
  discarded, or argued about with privacy.
allowed-tools: Read, Grep, Glob
```

# The agent read a card number because you asked it to

An agent run record is the richest telemetry in the estate and the most
dangerous to keep whole. The content it read — source files, tickets, documents
— lands in the record, so the record inherits every classification the source
had. Per-record retention forces a choice between losing the investigation and
keeping the data; per-field does not.

## When to use this

Before agent telemetry is retained at scale, and whenever a retention policy is
being written by someone who has not read a run record.

## Procedure

**1 — Read one full record.** Every step, every field. This is the step people
skip, and it is where the payment-card pattern in a legitimately-read source
file turns up.

**2 — Scan for sensitive patterns across all fields.** Card numbers, keys,
personal data, health terms. Record which field carried each hit — prompts and
tool results are the usual answer.

**3 — Classify fields by investigative value.** Timestamps, tool name, target
and verifier verdict answer most investigation questions. Prompts and raw tool
output answer few and carry most of the risk.

**4 — Set retention per field.** Long for the structural fields, short for the
content ones. Then age a record and check what an investigation could still do
with it — that check is what makes the policy defensible.

**5 — State what is lost.** A short prompt retention means you cannot re-derive
motivation after that window. Say so, rather than discovering it in an incident.

## Output contract

```json
{
  "record": {"steps": 0, "fields": ["str"]},
  "scan": [{"pattern": "str", "field": "str", "legitimate_source": "str"}],
  "retention": [{"field": "str", "days": 0, "investigative_value": "high|low"}],
  "aged_record": {"age_days": 0, "questions_still_answerable": ["str"], "lost": ["str"]}
}
```

## Failure modes

- **Retaining or discarding whole records.** Both answers are wrong.
- **Scanning prompts only.** Tool results carry the same content.
- **Not stating what the short windows lose.** That is the trade being made.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/detection/agent-telemetry-retention/scripts/agent_telemetry_retention.py
SCRIPT = "skills/detection/agent-telemetry-retention/scripts/agent_telemetry_retention.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; `sparse-checkout set skills` then materialises only what runs.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set", "skills"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

The full record contains a payment-card pattern found in a source file the agent read legitimately. Per-field retention keeps timestamps, tool, target and verifier for 400 days while dropping prompts at 30 days and tool results at 7, replacing them with hashes. After 90 days no sensitive content remains and the record can still answer what the agent did and what the harness believed.

## Your turn

Check the retention period on your agent traces. If it is the same as your firewall logs, one of those two numbers was chosen without anyone looking at what the traces contain.

---

**Next → [D1.6 · Distinguishing agent from human](https://spbreed.github.io/cyber-commons/lessons/D1.6.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/D1.5.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/D1.5.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*